In [1]:
!git clone https://github.com/samy1406/rag-pdf-chatbot.git
%cd rag-pdf-chatbot
!pip install -r requirements.txt
!pip install transformers accelerate bitsandbytes # Added for LLM inference
!mkdir -p data
!wget -O data/sample.pdf https://arxiv.org/pdf/1706.03762.pdf

Cloning into 'rag-pdf-chatbot'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 3), reused 15 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 7.31 KiB | 7.31 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/rag-pdf-chatbot
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 104.9 MB/s eta 0:00:00
   ━

In [2]:
!python src/ingest.py --file data/sample.pdf

Loading data/sample.pdf...
Splitting documents into chunks...
Generating embeddings and building vector database...
/content/rag-pdf-chatbot/src/ingest.py:39: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embedding = HuggingFaceEmbeddings(model_name = model_name)
modules.json: 100% 349/349 [00:00<00:00, 2.04MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 687kB/s]
README.md: 10.5kB [00:00, 32.3MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 330kB/s]
config.json: 100% 612/612 [00:00<00:00, 2.63MB/s]
model.safetensors: 100% 90.9M/90.9M [00:00<00:00, 110MB/s]
Loading weights: 100% 103/103 [00:00<00:00, 969.40it/s, Materializing param=pooler.den

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline, BitsAndBytesConfig
from langchain_huggingface import HuggingFacePipeline

print("Downloading and loading Zephyr-7B... (This will take 2-3 minutes)")
model_id = "HuggingFaceH4/zephyr-7b-beta"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# 1. Define the specific 4-bit quantization configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 2. Pass the config to the model loader
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)

# Create the text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
)
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [5]:
# 3. Create the Generation Pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=True,
)
llm = HuggingFacePipeline(pipeline=pipe)

# 4. Construct the Prompt Template (Zephyr specific format)
prompt_template = """<|system|>
You are a helpful technical assistant. Answer the question using ONLY the provided context. If the answer is not contained in the context, explicitly state that you do not know. Do not hallucinate external information.
</s>
<|user|>
Context: {context}

Question: {question}
</s>
<|assistant|>"""

prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])

def format_docs(docs):
    """Glues the retrieved chunks together into a single string."""
    return "\n\n".join(doc.page_content for doc in docs)

# 5. Build the LCEL Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Execute the Pipeline
question = "What is the architecture of the Transformer?"
print(f"\nQuestion: {question}")
print("Thinking...")

result = rag_chain.invoke(question)

# Parse out just the assistant's response from the full prompt output
final_answer = result.split("<|assistant|>")[-1].strip()

print("\n--- Final Answer ---")
print(final_answer)


Question: What is the architecture of the Transformer?
Thinking...


Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



--- Final Answer ---
The Transformer follows an overall architecture consisting of stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, as shown in Figure 1 (provided context). The encoder is composed of a stack of N = 6 identical layers, each with two sub-layers: a multi-head self-attention mechanism and a simple, position-wise fully connected feed-forward network. The decoder also contains a stack of N = 6 identical layers with the same sub-layers. In the encoder, all sub-layers and embedding layers produce outputs of dimension dmodel = 512. The Transformer uses multi-head attention in three different ways: encoder-decoder attention, self-attention in the encoder, and self-attention in the decoder. Residual dropout is applied to the output of each sub-layer and to the sums of the embeddings and the positional encodings in both the encoder and decoder stacks, with a rate of Pdrop = 0.1 for the base model.
